In [ ]:
import os,subprocess,sys,hashlib,base64,zlib,json
from pathlib import Path
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["BLUFF_FAST_OBSERVATIONS"]="1"
work=Path("/kaggle/working/training");work.mkdir(exist_ok=True)
sources={'bluff_policy.py': '"""Compact learned Bluff action scorer using own cards and public evidence."""\nfrom collections import Counter, defaultdict\nimport json\nfrom pathlib import Path\n\nRANKS = ("A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K")\nPLACE_FEATURES = ["batch_size", "truthful", "bluff", "bluff_pile", "bluff_call_rate",\n                  "bluff_finish", "remaining_groups", "remaining_largest_group", "claimed_rank_held"]\nCHALLENGE_FEATURES = ["bias", "revealed_bluff_rate", "rank_held", "claim_count", "pile_size",\n                      "opponent_finishing", "own_small_hand", "opponent_small_hand", "samples"]\nINITIAL_WEIGHTS = [4., 5., -2., -4., -8., -8., -1., .2, .5,\n                   -3.5, 7., .7, -.3, -.5, 1., -1., .5, 0.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(INITIAL_WEIGHTS):\n            raise ValueError("wrong feature count")\n        self.weights = [float(v) for v in weights]\n        self.processed = 0\n        self.claims = {}\n        self.stats = defaultdict(lambda: [0, 0])\n        self.our_outcomes = [0, 0]\n\n    def update(self, observation):\n        history = observation.get("public_history", [])\n        if len(history) < self.processed:\n            self.processed, self.claims, self.stats, self.our_outcomes = 0, {}, defaultdict(lambda: [0, 0]), [0, 0]\n        for event in history[self.processed:]:\n            kind = event.get("type")\n            identity = event.get("claim_id")\n            if kind == "claim":\n                self.claims[identity] = (event.get("seat"), event.get("count", 1))\n            elif kind in ("claim_accepted", "challenge_resolved"):\n                placer, count = self.claims.pop(identity, (event.get("placer", event.get("seat")), event.get("count", 1)))\n                if placer == observation["seat"]:\n                    self.our_outcomes[0] += 1\n                    self.our_outcomes[1] += int(kind == "challenge_resolved")\n                if kind == "challenge_resolved" and isinstance(event.get("truthful"), bool):\n                    stats = self.stats[(placer, count)]\n                    stats[0] += 1\n                    stats[1] += int(not event["truthful"])\n        self.processed = len(history)\n\n    def candidates(self, observation):\n        hand = observation["hand"]\n        held = Counter(c["rank"] for c in hand)\n        pile = min(observation.get("pile_size", 0), 40) / 20.\n        if observation["phase"] == "CHALLENGE":\n            claim = observation["pending_claim"]\n            count, rank, placer = claim["count"], claim["claimed_rank"], claim["seat"]\n            copies = 4 * observation.get("game_config", {}).get("decks", 1)\n            if count > copies - held[rank]:\n                return [({"type": "challenge"}, [0.] * len(self.weights))]\n            samples, bluffs = self.stats[(placer, count)]\n            posterior = (bluffs + .6) / (samples + 5.)\n            sizes = observation["hand_sizes"]\n            vector = [1., posterior, held[rank] / copies, count / 4., pile,\n                      float(claim.get("provisional_finish", False)), float(len(hand) <= 3),\n                      1. / max(1, sizes[placer]), min(samples, 20) / 20.]\n            return [({"type": "accept"}, [0.] * len(self.weights)),\n                    ({"type": "challenge"}, [0.] * len(PLACE_FEATURES) + vector)]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card["rank"]].append(card)\n        actions = []\n        # Safe truthful groups are always available. Constructed bluff candidates\n        # use only currently owned references; the count is the actual batch size.\n        for rank, cards in groups.items():\n            actions.append({"type": "place", "card_refs": [c["card_ref"] for c in cards[:4]], "claimed_rank": rank})\n        ordered = sorted(hand, key=lambda c: (held[c["rank"]], RANKS.index(c["rank"])))\n        for count in (2, 3):\n            if len(ordered) >= count:\n                for rank in groups:\n                    actions.append({"type": "place", "card_refs": [c["card_ref"] for c in ordered[:count]], "claimed_rank": rank})\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        by_ref = {c["card_ref"]: c for c in hand}\n        result = []\n        for action in actions:\n            refs = set(action["card_refs"])\n            rank = action["claimed_rank"]\n            truthful = all(by_ref[ref]["rank"] == rank for ref in refs)\n            remaining = Counter(c["rank"] for c in hand if c["card_ref"] not in refs)\n            vector = [len(refs), float(truthful), float(not truthful), float(not truthful) * pile,\n                      float(not truthful) * call_rate, float(not truthful and len(refs) == len(hand)),\n                      len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES)))\n        return result\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda candidate:\n                   sum(w * value for w, value in zip(self.weights, candidate[1])))[0]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-bluff-linear-v1", "features": PLACE_FEATURES + CHALLENGE_FEATURES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'bluff_policy_v2.py': '"""Learned Bluff scorer with conservative rank bounds and public caller evidence.\n\nInference reads only own hand/private history and public events. Unknown discarded\ncards decrement every rank bound, ensuring no hidden-hand identity is assumed.\n"""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom bluff_policy import Policy as V1, PLACE_FEATURES, CHALLENGE_FEATURES, INITIAL_WEIGHTS as V1_INITIAL, RANKS\n\nEXTRA_PLACE = [\'bluff_any_call_probability\', \'bluff_impossible_for_caller\',\n               \'truthful_turns_removed\', \'bluff_known_pile_fraction\',\n               \'truthful_finish\', \'bluff_no_impossibility_call_possible\']\nEXTRA_CHALLENGE = [\'placer_global_revealed_bluff_rate\', \'known_outside_rank_fraction\',\n                   \'known_pile_fraction\', \'claim_fraction_of_original_hand\',\n                   \'finishing_large_batch\', \'known_placer_rank_before_claim\']\nFEATURE_NAMES = PLACE_FEATURES + CHALLENGE_FEATURES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = [.5, 1., -.5, -2., -8., -3., -1., .2, .5] + V1_INITIAL[len(PLACE_FEATURES):] + [-6., -12., 3., -1., 4., 3., 1., 1., -.5, .5, .5, -.5]\n\n\ndef identity(card):\n    return card.get(\'deck\', 0), card[\'rank\'], card[\'suit\']\n\n\nclass Policy(V1):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid model weights\')\n        super().__init__(weights[:len(V1_INITIAL)])\n        self.weights = list(map(float, weights))\n        self._reset_evidence()\n\n    def _reset_evidence(self):\n        self.folded_history = []\n        self.known = defaultdict(Counter)\n        self.pile_known = {}\n        self.calls = defaultdict(lambda: [0, 0])\n        self.global_bluffs = defaultdict(lambda: [0, 0])\n        self.pending_meta = {}\n        self.active = []\n        self.before_claim = {}\n\n    def update(self, observation):\n        history = observation.get(\'public_history\', [])\n        if len(history) < len(self.folded_history) or history[:len(self.folded_history)] != self.folded_history:\n            self.processed = 0\n            self.claims = {}\n            self.stats = defaultdict(lambda: [0, 0])\n            self.our_outcomes = [0, 0]\n            self._reset_evidence()\n        super().update(observation)\n        own_places = {event[\'claim_id\']: event[\'cards\'] for event in observation.get(\'private_history\', []) if event.get(\'type\') == \'own_placement\'}\n        for event in history[len(self.folded_history):]:\n            kind, cid = event.get(\'type\'), event.get(\'claim_id\')\n            if kind == \'game_start\':\n                self.active = list(event[\'active_seats\'])\n            elif kind == \'claim\':\n                placer, count = event[\'seat\'], event[\'count\']\n                self.before_claim[cid] = dict(self.known[placer])\n                # The rank in an unverified claim is not evidence of actual cards.\n                for rank in list(self.known[placer]):\n                    self.known[placer][rank] = max(0, self.known[placer][rank] - count)\n                priority = sorted((s for s in self.active if s != placer), key=lambda s: (s - placer) % observation[\'game_config\'][\'players\'])\n                self.pending_meta[cid] = (placer, count, priority, bool(event.get(\'provisional_finish\')))\n                for card in own_places.get(cid, []):\n                    self.pile_known[identity(card)] = card\n            elif kind in (\'claim_accepted\', \'challenge_resolved\'):\n                placer, count, priority, finish = self.pending_meta.pop(cid)\n                selected = event.get(\'challenger\') if kind == \'challenge_resolved\' else None\n                # The first selected challenger is revealed. Earlier seats must\n                # have accepted; later choices remain unknown and are not counted.\n                for seat in priority:\n                    record = self.calls[(seat, count, finish)]\n                    record[0] += 1\n                    record[1] += int(seat == selected)\n                    if seat == selected:\n                        break\n                if kind == \'challenge_resolved\':\n                    record = self.global_bluffs[placer]\n                    record[0] += 1\n                    record[1] += int(not event[\'truthful\'])\n                    for card in event[\'revealed_batch\']:\n                        self.pile_known[identity(card)] = card\n            elif kind == \'pickup\':\n                recipient = event[\'recipient\']\n                self.known[recipient].update(card[\'rank\'] for card in self.pile_known.values())\n                self.pile_known = {}\n            elif kind == \'finish\':\n                self.active = [s for s in self.active if s != event[\'seat\']]\n        self.folded_history = list(history)\n\n    def caller_probability(self, observation, count, finish):\n        escape = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            # Beta prior avoids certainties from small or selectively observed samples.\n            escape *= 1. - (calls + 1.) / (samples + 5.)\n        return 1. - escape\n\n    def candidates(self, observation):\n        hand = observation[\'hand\']\n        held = Counter(c[\'rank\'] for c in hand)\n        copies = 4 * observation.get(\'game_config\', {}).get(\'decks\', 1)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            placer, rank, count = claim[\'seat\'], claim[\'claimed_rank\'], claim[\'count\']\n            outside = held[rank] + sum(counts[rank] for seat, counts in self.known.items() if seat not in (observation[\'seat\'], placer))\n            if count > copies - outside:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n            base = super().candidates(observation)\n            samples, bluffs = self.global_bluffs.get(placer, (0, 0))\n            extra = [(bluffs + .6) / (samples + 5.), outside / copies,\n                     len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     count / max(1, observation[\'hand_sizes\'][placer] + count),\n                     float(claim.get(\'provisional_finish\', False)) * count / 4.,\n                     self.before_claim.get(claim[\'claim_id\'], {}).get(rank, 0) / copies]\n            return [(a, v + [0.] * len(EXTRA_PLACE) + (extra if a[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE))) for a, v in base]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card[\'rank\']].append(card)\n        actions = [action for action, _ in super().candidates(observation)]\n        # Full truthful batches plus partial truthful batches and mixed shedding\n        # batches of up to four cards, including a possible finishing bluff.\n        for rank, cards in groups.items():\n            for count in range(1, min(4, len(cards)) + 1):\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:count]], \'claimed_rank\': rank})\n        ordered = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))\n        for count in range(2, min(4, len(hand)) + 1):\n            for rank in RANKS:\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in ordered[:count]], \'claimed_rank\': rank})\n        by_ref, seen, result = {c[\'card_ref\']: c for c in hand}, set(), []\n        pile = min(observation.get(\'pile_size\', 0), 40) / 20.\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        for action in actions:\n            refs, rank = set(action[\'card_refs\']), action[\'claimed_rank\']\n            key = tuple(sorted(refs)), rank\n            if key in seen:\n                continue\n            seen.add(key)\n            truthful = all(by_ref[r][\'rank\'] == rank for r in refs)\n            bluff = float(not truthful)\n            remaining = Counter(c[\'rank\'] for c in hand if c[\'card_ref\'] not in refs)\n            finish = not remaining\n            vector = [len(refs), float(truthful), bluff, bluff * pile, bluff * call_rate,\n                      float(bluff and finish), len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            impossible = any(self.known.get(s, {}).get(rank, 0) + len(refs) > copies for s in observation[\'active_seats\'] if s != observation[\'seat\'])\n            extra = [bluff * self.caller_probability(observation, len(refs), finish),\n                     bluff * impossible, sum((n + 3) // 4 for n in held.values()) - sum((n + 3) // 4 for n in remaining.values()),\n                     bluff * len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     float(truthful and finish), bluff * float(held[rank] >= len(refs))]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES) + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return result\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v2\', \'features\': FEATURE_NAMES,\n                                        \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v3.py': '"""Bluff v2 model with additional, sound public/own pile rank deductions.\n\nOlder unchallenged batches are never assumed truthful. These deductions use\nonly own placement cards and the last publicly revealed batch before pickup.\n"""\nfrom collections import Counter\nfrom bluff_policy_v2 import Policy as V2, FEATURE_NAMES, INITIAL_WEIGHTS\n\n\nclass Policy(V2):\n    def candidates(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            rank, count, placer = claim[\'claimed_rank\'], claim[\'count\'], claim[\'seat\']\n            # Known old pile cards, current own cards and lower bounds on other\n            # players\' hands are disjoint, so their sum is a conservative count.\n            pile_count = Counter(card[\'rank\'] for card in self.pile_known.values())[rank]\n            held_count = sum(card[\'rank\'] == rank for card in observation[\'hand\'])\n            other_count = sum(counts[rank] for seat, counts in self.known.items()\n                              if seat not in (observation[\'seat\'], placer))\n            copies = 4 * observation[\'game_config\'][\'decks\']\n            if count > copies - held_count - other_count - pile_count:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n        return super().candidates(observation)\n', 'bluff_policy_v4.py': '"""Linear Bluff scorer with learned interactions for public call-risk adaptation."""\nfrom collections import Counter\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v2 import FEATURE_NAMES as V2_NAMES, INITIAL_WEIGHTS as V2_INITIAL\n\nEXTRA_PLACE = [\'bluff_call_probability_batch\', \'bluff_call_probability_pile\',\n               \'bluff_call_probability_finish\', \'bluff_nearest_opponent_finish\',\n               \'truthful_call_probability_batch\', \'bluff_remaining_truthful_turns\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_finishing\', \'count_revealed_bluff_pile\',\n                   \'global_revealed_bluff_pile\', \'count_revealed_bluff_small_own_hand\',\n                   \'global_revealed_bluff_count\', \'opponent_hand_pace_relative_to_own\']\nFEATURE_NAMES = V2_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V2_INITIAL + [0.] * 12\n\n\nclass Policy(V3):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v4 model weights\')\n        super().__init__(weights[:len(V2_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        hand = observation[\'hand\']\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            global_rate = (bluffs + .6) / (samples + 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            count_rate = (bluffs + .6) / (samples + 5.)\n            extra = [global_rate * float(claim.get(\'provisional_finish\', False)),\n                     count_rate * pile, global_rate * pile,\n                     count_rate * float(len(hand) <= 3), global_rate * claim[\'count\'] / 4.,\n                     (len(hand) - observation[\'hand_sizes\'][claim[\'seat\']]) / max(1, len(hand))]\n            return [(action, vector[:len(V2_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     (extra if action[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE)))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in hand}\n        held = Counter(card[\'rank\'] for card in hand)\n        nearest = min(observation[\'hand_sizes\'][seat] for seat in observation[\'active_seats\']\n                      if seat != observation[\'seat\'])\n        results = []\n        risks = {}\n        for action, vector in candidates:\n            count = len(action[\'card_refs\']); finish = count == len(hand)\n            if count not in risks:\n                risks[count] = self.caller_probability(observation, count, finish)\n            risk = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            remaining = held.copy()\n            for ref in action[\'card_refs\']:\n                remaining[by_ref[ref][\'rank\']] -= 1\n            turns = sum((n + 3) // 4 for n in remaining.values() if n > 0)\n            extra = [bluff * risk * count / 4., bluff * risk * pile,\n                     bluff * risk * float(finish), bluff / max(1, nearest),\n                     (1. - bluff) * risk * count / 4., bluff * turns / 13.]\n            results.append((action, vector[:len(V2_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v4\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v5.py': '"""Public caller pooling and uncertainty features for a learned Bluff scorer.\n\nBeta summaries are modeling features, not calibrated certainty claims. Observed\nselected/earlier caller outcomes are pooled; later sealed votes stay unknown.\n"""\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v4 import Policy as V4, FEATURE_NAMES as V4_NAMES, INITIAL_WEIGHTS as V4_INITIAL\n\nEXTRA_PLACE = [\'bluff_pooled_caller_probability\', \'bluff_pooled_caller_uncertainty\',\n               \'bluff_specific_caller_uncertainty\', \'bluff_sparse_own_outcomes\',\n               \'truthful_pooled_caller_probability_batch\', \'bluff_pooled_caller_probability_pile\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_uncertainty\', \'count_revealed_bluff_uncertainty\']\nFEATURE_NAMES = V4_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V4_INITIAL + [0.] * 8\n\n\ndef beta_summary(samples, successes, prior_success=1., prior_total=5.):\n    a = successes + prior_success\n    b = samples - successes + prior_total - prior_success\n    return a / (a + b), math.sqrt(a * b / ((a + b) ** 2 * (a + b + 1.)))\n\n\nclass Policy(V4):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v5 model weights\')\n        super().__init__(weights[:len(V4_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def caller_summaries(self, observation, count=None, finish=None):\n        escape_mean = escape_upper = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            if count is None:\n                samples = calls = 0\n                for (caller, _, _), record in self.calls.items():\n                    if caller == seat:\n                        samples += record[0]; calls += record[1]\n            else:\n                samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            mean, deviation = beta_summary(samples, calls)\n            escape_mean *= 1. - mean\n            escape_upper *= 1. - min(1., mean + deviation)\n        return 1. - escape_mean, escape_mean - escape_upper\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            _, global_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            _, count_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            return [(action, vector[:len(V4_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     ([global_uncertainty, count_uncertainty] if action[\'type\'] == \'challenge\' else [0., 0.]))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in observation[\'hand\']}\n        pooled, pooled_uncertainty = self.caller_summaries(observation)\n        sparse = float(self.our_outcomes[0] <= 2)\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        risks = {}\n        results = []\n        for action, vector in candidates:\n            count = len(action[\'card_refs\'])\n            if count not in risks:\n                risks[count] = self.caller_summaries(observation, count, count == len(observation[\'hand\']))\n            _, specific_uncertainty = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            extra = [bluff * pooled, bluff * pooled_uncertainty, bluff * specific_uncertainty,\n                     bluff * sparse, (1. - bluff) * pooled * count / 4., bluff * pooled * pile]\n            results.append((action, vector[:len(V4_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v5\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_mlp_policy.py': '"""Tiny residual neural Bluff scorer with standard-library inference."""\nimport math\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\n\n\nclass Policy(Features):\n    def __init__(self, document):\n        if document.get(\'format\') != \'triad-bluff-residual-mlp-v1\' or document.get(\'features\') != FEATURE_NAMES:\n            raise ValueError(\'incompatible neural Bluff schema\')\n        base = document[\'base_weights\']\n        super().__init__(base)\n        self.hidden = document[\'hidden_weight\']; self.bias = document[\'hidden_bias\']\n        self.output = document[\'output_weight\']; self.output_bias = float(document[\'output_bias\'])\n        self.means = document[\'feature_mean\']; self.scales = document[\'feature_scale\']\n        width = len(self.hidden)\n        if not 1 <= width <= 128 or len(self.bias) != width or len(self.output) != width:\n            raise ValueError(\'invalid hidden layer shape\')\n        if any(len(row) != len(FEATURE_NAMES) for row in self.hidden) or len(self.means) != len(FEATURE_NAMES) or len(self.scales) != len(FEATURE_NAMES):\n            raise ValueError(\'invalid neural input shape\')\n        parameters = base + self.bias + self.output + self.means + self.scales + [self.output_bias]\n        parameters += [weight for row in self.hidden for weight in row]\n        if not all(math.isfinite(float(weight)) for weight in parameters) or any(scale <= 0. for scale in self.scales):\n            raise ValueError(\'nonfinite neural parameters or invalid feature scale\')\n\n    def score_features(self, features):\n        normalized = [(value - mean) / scale for value, mean, scale in zip(features, self.means, self.scales)]\n        residual = self.output_bias\n        for row, bias, output in zip(self.hidden, self.bias, self.output):\n            hidden = max(0., bias + sum(weight * value for weight, value in zip(row, normalized)))\n            residual += output * hidden\n        return sum(weight * value for weight, value in zip(self.weights, features)) + residual\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda item: self.score_features(item[1]))[0]\n\n\nclass FastPolicy(Policy):\n    """Training-only NumPy batching; near ties use exact deployment scoring."""\n    def __init__(self, document):\n        super().__init__(document)\n        import numpy as np\n        self.np = np\n        self.matrix = np.asarray(self.hidden, dtype=np.float64)\n        self.hidden_bias = np.asarray(self.bias, dtype=np.float64)\n        self.output_array = np.asarray(self.output, dtype=np.float64)\n        self.base_array = np.asarray(self.weights, dtype=np.float64)\n        self.mean_array = np.asarray(self.means, dtype=np.float64)\n        self.scale_array = np.asarray(self.scales, dtype=np.float64)\n        # These exported parameters are immutable configuration, never actor\n        # memory. Freezing them allows exact snapshot sharing during rollouts.\n        self.weights = tuple(self.weights)\n        self.hidden = tuple(tuple(row) for row in self.hidden)\n        self.bias = tuple(self.bias); self.output = tuple(self.output)\n        self.means = tuple(self.means); self.scales = tuple(self.scales)\n        for matrix in (self.matrix, self.hidden_bias, self.output_array,\n                       self.base_array, self.mean_array, self.scale_array):\n            matrix.flags.writeable = False\n\n\n    def __deepcopy__(self, memo):\n        # Copy public/private actor memory, and share only frozen configuration.\n        # This avoids repeatedly copying thousands of constant network weights.\n        import copy\n        clone = type(self).__new__(type(self))\n        memo[id(self)] = clone\n        configuration = {\'np\', \'weights\', \'hidden\', \'bias\', \'output\', \'output_bias\',\n                         \'means\', \'scales\', \'matrix\', \'hidden_bias\', \'output_array\',\n                         \'base_array\', \'mean_array\', \'scale_array\'}\n        for name, value in self.__dict__.items():\n            setattr(clone, name, value if name in configuration else copy.deepcopy(value, memo))\n        return clone\n\n    def act(self, observation):\n        self.update(observation)\n        candidates = self.candidates(observation)\n        matrix = self.np.asarray([vector for _, vector in candidates], dtype=self.np.float64)\n        scores = matrix @ self.base_array + self.output_bias\n        scores += self.np.maximum(0., ((matrix - self.mean_array) / self.scale_array) @ self.matrix.T + self.hidden_bias) @ self.output_array\n        largest = float(scores.max())\n        band = 1e-8 * max(1., abs(largest))\n        finalists = [index for index, value in enumerate(scores) if value >= largest - band]\n        # Feature aliases have exactly the same stdlib score. Memoize within\n        # this decision only; max still retains the first action on exact ties.\n        exact_scores = {}\n        def exact_score(index):\n            vector = candidates[index][1]\n            identity = tuple(vector)\n            if identity not in exact_scores:\n                exact_scores[identity] = self.score_features(vector)\n            return exact_scores[identity]\n        chosen = max(finalists, key=exact_score)\n        return candidates[chosen][0]\n', 'bluff_training_observation.py': '"""Training-only observation shortcut for verified read-only simulator actors.\n\nAll fields still come from the SDK\'s filtered observe(). The engine is not\nadvanced while histories are temporarily detached. Only the requested seat\'s\nprivate history is returned. Never import this helper into an inference image.\n"""\nfrom triad_bluff import engine\n\n\ndef observe_read_only(state, seat):\n    public = state.public_history\n    private = state.private_history[seat]\n    try:\n        state.public_history = []\n        state.private_history[seat] = []\n        observation = engine.observe(state, seat)\n    finally:\n        state.public_history = public\n        state.private_history[seat] = private\n    observation[\'public_history\'] = public\n    observation[\'private_history\'] = private\n    return observation\n', 'train_bluff_league.py': '"""Train against diverse Bluff policies with paired seat blocks and held-out tests."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get(\'BLUFF_SDK\', ROOT.parent / \'bluffSDK\'))\nsys.path.insert(0, str(SDK / \'local_evaluator/src\'))\nfrom triad_bluff import engine\nfrom triad_bluff.baselines import HeuristicAgent, RandomAgent, TruthfulAgent, AggressiveChallengeAgent\nfrom bluff_policy import Policy as V1, INITIAL_WEIGHTS as V1_INITIAL, RANKS\nfrom bluff_policy_v2 import Policy as V2, INITIAL_WEIGHTS, FEATURE_NAMES\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v4 import Policy as V4, INITIAL_WEIGHTS as V4_INITIAL\nfrom bluff_policy_v5 import Policy as V5, INITIAL_WEIGHTS as V5_INITIAL\nfrom bluff_training_observation import observe_read_only\n\n\nclass Starter(TruthfulAgent):\n    def act(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            return super().act(observation)\n        from collections import defaultdict\n        groups = defaultdict(list)\n        for card in observation[\'hand\']:\n            groups[card[\'rank\']].append(card)\n        rank, cards = max(groups.items(), key=lambda item: (len(item[1]), item[0]))\n        return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:4]], \'claimed_rank\': rank}\n\n\nclass SDKCompetitive:\n    """The SDK\'s supplied adaptive strategy, used only as a development rival."""\n    def __init__(self):\n        import importlib.util\n        path = SDK / \'agents/bluff-agent/src/strategy.py\'\n        spec = importlib.util.spec_from_file_location(\'sdk_competitive_strategy\', path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        self.select_action = module.select_action\n        self.state = None\n\n    def act(self, observation):\n        action, self.state = self.select_action(observation, self.state)\n        return action\n\n\nclass StochasticBluffer(TruthfulAgent):\n    def __init__(self, seed, bluff_rate, call_rate):\n        self.rng = random.Random(seed)\n        self.bluff_rate, self.call_rate = bluff_rate, call_rate\n\n    def act(self, observation):\n        truthful = super().act(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if truthful[\'type\'] == \'challenge\':\n                return truthful\n            finishing = observation[\'pending_claim\'].get(\'provisional_finish\', False)\n            probability = min(.95, self.call_rate + .2 * finishing)\n            return {\'type\': \'challenge\' if self.rng.random() < probability else \'accept\'}\n        hand = observation[\'hand\']\n        if len(hand) > 1 and self.rng.random() < self.bluff_rate:\n            from collections import Counter\n            held = Counter(c[\'rank\'] for c in hand)\n            count = min(4, len(hand)) if len(hand) <= 4 else self.rng.choice((2, 3, 4))\n            chosen = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))[:count]\n            # Owning copies of the claimed rank makes impossible calls less likely.\n            claimed = max(RANKS, key=lambda rank: held[rank])\n            return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in chosen], \'claimed_rank\': claimed}\n        return truthful\n\n\ndef make_policy(spec, seed):\n    kind = spec[\'kind\']\n    if kind == \'mlp\':\n        from bluff_mlp_policy import FastPolicy\n        return FastPolicy(spec[\'model\'])\n    if kind == \'v1\':\n        return V1(spec[\'weights\'])\n    if kind == \'v2\':\n        return V2(spec[\'weights\'])\n    if kind == \'v3\':\n        return V3(spec[\'weights\'])\n    if kind == \'v4\':\n        return V4(spec[\'weights\'])\n    if kind == \'v5\':\n        return V5(spec[\'weights\'])\n    if kind == \'bluffer\':\n        return StochasticBluffer(seed, spec[\'bluff_rate\'], spec[\'call_rate\'])\n    if kind == \'starter\':\n        return Starter()\n    if kind == \'competitive\':\n        return SDKCompetitive()\n    if kind == \'truthful\':\n        return TruthfulAgent()\n    if kind == \'random\':\n        return RandomAgent(seed)\n    if kind == \'aggressive\':\n        return AggressiveChallengeAgent(seed)\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals, horizon=1000):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, rival in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(rival, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=horizon), seed)\n    while state.outcome is None and seat not in state.finished_set:\n        decision = engine.next_decision(state)\n        observation = observe_read_only(state, decision.seat) if os.environ.get(\'BLUFF_FAST_OBSERVATIONS\') == \'1\' else engine.observe(state, decision.seat)\n        payload = actors[decision.seat].act(observation)\n        action = engine.decode_action(state, decision, payload)\n        if not engine.step(state, decision, action).ok:\n            raise RuntimeError(\'policy produced an illegal action\')\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    return state.outcome.points[seat], 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode, *options = job\n    horizon = options[0] if options else 1000\n    points, wins = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x424C554646)\n        if mode == \'league\':\n            rivals = [rng.choice(pool) for _ in range(3)]\n        elif mode == \'bluffer\':\n            rivals = [{\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}] * 3\n        else:\n            rivals = [{\'kind\': mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals, horizon) for seat in range(4)]\n        points.append(sum(p for p, _ in outcomes) / 4.)\n        wins.append(sum(w for _, w in outcomes) / 4.)\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(seeds), \'block_points\': points, \'block_wins\': wins}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith(\'block_\')}\n\n\ndef parallel_evaluate(pool, candidate, seeds, reference, mode):\n    """Distribute validation seed blocks instead of leaving other workers idle."""\n    chunks = [seeds[i:i + 16] for i in range(0, len(seeds), 16)]\n    parts = list(pool.map(evaluate_job, [(candidate, chunk, reference, mode) for chunk in chunks]))\n    points = [point for part in parts for point in part[\'block_points\']]\n    wins = [win for part in parts for win in part[\'block_wins\']]\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(points), \'block_points\': points, \'block_wins\': wins}\n\n\ndef paired_interval(candidate, reference, seed=20261014):\n    differences = np.asarray(candidate[\'block_points\']) - np.asarray(reference[\'block_points\'])\n    rng = np.random.default_rng(seed)\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {\'mean_gain\': float(differences.mean()), \'ci95\': [float(v) for v in np.quantile(means, [.025, .975])],\n            \'unit\': \'points per game\', \'sampling_unit\': \'four-seat seed block\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path)\n    parser.add_argument(\'--reference-policy\', type=Path)\n    parser.add_argument(\'--policy-kind\', choices=(\'v2\', \'v3\', \'v4\', \'v5\'), default=\'v2\')\n    parser.add_argument(\'--fast-observations\', action=\'store_true\')\n    parser.add_argument(\'--stress-opponents\', action=\'store_true\')\n    parser.add_argument(\'--selection-seed-start\', type=int, default=8_000_000)\n    parser.add_argument(\'--test-seed-start\', type=int, default=9_000_000)\n    parser.add_argument(\'--train-seed-start\', type=int, default=50_000)\n    parser.add_argument(\'--iterations\', type=int, default=30)\n    parser.add_argument(\'--population\', type=int, default=32)\n    parser.add_argument(\'--train-blocks\', type=int, default=8)\n    parser.add_argument(\'--train-horizon\', type=int, default=200)\n    parser.add_argument(\'--validation-blocks\', type=int, default=64)\n    parser.add_argument(\'--test-blocks\', type=int, default=256)\n    parser.add_argument(\'--workers\', type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument(\'--seed\', type=int, default=20261016)\n    parser.add_argument(\'--output\', type=Path, default=ROOT / \'bluff-league-output\')\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers, args.train_horizon) < 1 or args.train_horizon > 1000:\n        parser.error(\'counts must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    if args.fast_observations:\n        os.environ[\'BLUFF_FAST_OBSERVATIONS\'] = \'1\'\n    incumbent_document = json.loads(args.incumbent.read_text()) if args.incumbent else {\'weights\': V1_INITIAL}\n    weights = incumbent_document[\'weights\']\n    incumbent_kind = incumbent_document.get(\'training\', {}).get(\'policy_kind\', \'v2\' if len(weights) == len(FEATURE_NAMES) else \'v4\' if len(weights) == len(V4_INITIAL) else \'v5\' if len(weights) == len(V5_INITIAL) else \'v1\')\n    incumbent = {\'kind\': incumbent_kind, \'weights\': weights}\n    initial_weights = V5_INITIAL if args.policy_kind == \'v5\' else V4_INITIAL if args.policy_kind == \'v4\' else INITIAL_WEIGHTS\n    mean = np.asarray(weights + initial_weights[len(weights):], dtype=float)\n    best, sigma = mean.copy(), np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    cautious = list(INITIAL_WEIGHTS); cautious[2] -= 8.; cautious[4] -= 8.\n    daring = list(INITIAL_WEIGHTS); daring[2] += 4.; daring[5] += 4.\n    reference = [{\'kind\': k} for k in (\'starter\', \'heuristic\', \'truthful\', \'aggressive\', \'random\', \'competitive\')] + [\n        incumbent, {\'kind\': \'v2\', \'weights\': INITIAL_WEIGHTS}, {\'kind\': \'v2\', \'weights\': cautious},\n        {\'kind\': \'v2\', \'weights\': daring}] + [\n        {\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n    if args.reference_policy:\n        reference_document = json.loads(args.reference_policy.read_text())\n        reference.append({\'kind\': reference_document.get(\'training\', {}).get(\'policy_kind\', \'v1\' if len(reference_document[\'weights\']) == len(V1_INITIAL) else \'v2\'),\n                          \'weights\': reference_document[\'weights\']})\n    if args.stress_opponents:\n        reference.extend({\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c}\n                         for b, c in ((.1, .35), (.65, .35), (.95, .0), (.95, .35)))\n    archive = list(reference)\n    selection_seeds = list(range(args.selection_seed_start, args.selection_seed_start + args.validation_blocks))\n    history, started = [], time.time()\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        baseline = parallel_evaluate(pool, incumbent, selection_seeds, reference, \'league\')\n        initial = parallel_evaluate(pool, {\'kind\': args.policy_kind, \'weights\': best.tolist()}, selection_seeds, reference, \'league\')\n        selected_score = initial[\'points\']\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(args.train_seed_start + generation * args.train_blocks, args.train_seed_start + (generation + 1) * args.train_blocks))\n            # Fast training probes use a shorter cap; model selection and final\n            # reporting always use the real 1,000-turn cap. Short-cap results are\n            # approximate training rewards, never evidence of event performance.\n            scores = list(pool.map(evaluate_job, [({\'kind\':args.policy_kind,\'weights\':w.tolist()},seeds,archive,\'league\',args.train_horizon) for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i][\'points\'], scores[i][\'win_rate\']), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {\'kind\':args.policy_kind,\'weights\':candidate.tolist()}\n            validation = parallel_evaluate(pool, spec, selection_seeds, reference, \'league\')\n            if validation[\'points\'] > selected_score:\n                best, selected_score = candidate.copy(), validation[\'points\']\n            if generation % 4 == 3:\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {\'generation\':generation+1,\'train\':summarize(scores[ranked[0]]),\'validation\':summarize(validation),\n                      \'paired_validation_gain\':paired_interval(validation,baseline),\'selected_points\':selected_score,\n                      \'archive_size\':len(archive),\'elapsed_seconds\':round(time.time()-started,1)}\n            history.append(record); print(json.dumps(record),flush=True)\n            policy_class = V5 if args.policy_kind == \'v5\' else V4 if args.policy_kind == \'v4\' else V2\n            policy_class(best).save(args.output/\'bluff-policy-v2.json\',{\'method\':\'archive-league cross-entropy policy optimization\',\n                \'fast_observations\': args.fast_observations, \'stress_opponents\': args.stress_opponents,\n                \'policy_kind\': args.policy_kind, \'selection_seed_start\': args.selection_seed_start,\n                \'test_seed_start\': args.test_seed_start, \'train_seed_start\': args.train_seed_start,\n                \'seed\':args.seed,\'generations\':generation+1,\'population\':args.population,\n                \'train_blocks\':args.train_blocks,\'train_horizon\':args.train_horizon,\'evaluation_horizon\':1000,\'validation_blocks\':args.validation_blocks,\n                \'selected_validation_points\':selected_score,\'incumbent_validation_points\':baseline[\'points\'],\n                \'sdk_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\'})\n            (args.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n            (args.output/\'archive.json\').write_text(json.dumps(archive,indent=2)+\'\\n\')\n        seeds = list(range(args.test_seed_start,args.test_seed_start+args.test_blocks))\n        chosen = {\'kind\':args.policy_kind,\'weights\':best.tolist()}\n        modes = (\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\',\'competitive\')\n        results = list(pool.map(evaluate_job,[(spec,seeds,reference,mode) for mode in modes for spec in (chosen,incumbent)]))\n        report = {mode:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\n                        \'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,mode in enumerate(modes)}\n        (args.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        print(json.dumps({\'held_out_evaluation\':report}),flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'collect_bluff_counterfactual.py': '"""Paired offline policy-improvement returns; deployment features are own/public.\n\nThe dealt simulator hands are held fixed within each labeled state. Opponent\nrandom choices are averaged using common trial seeds across candidate actions.\nWhole-game, phase-stratified reservoir sampling covers later/endgame decisions.\nThere is no draw deck to shuffle in Bluff. Hidden simulator state is confined to\ntraining returns and never appears in the exported feature vectors.\n"""\nimport argparse\nimport concurrent.futures\nimport copy\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport time\nfrom collections import Counter\nimport numpy as np\nfrom train_bluff_league import engine, make_policy\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\nfrom bluff_training_observation import observe_read_only\n\n\ndef reward(state, seat):\n    if seat in state.finished_order:\n        return float(state.config.position_points[state.finished_order.index(seat)])\n    return float(state.outcome.points[seat])\n\n\ndef advance(state, actors):\n    decision = engine.next_decision(state)\n    observation = observe_read_only(state, decision.seat)\n    action = actors[decision.seat].act(observation)\n    if not engine.step(state, decision, engine.decode_action(state, decision, action)).ok:\n        raise RuntimeError(\'rollout actor produced an illegal action\')\n\n\ndef copy_snapshot(snapshot):\n    if os.environ.get(\'BLUFF_FAST_SNAPSHOTS\') == \'1\':\n        from bluff_training_snapshot import clone_snapshot\n        return clone_snapshot(snapshot)\n    return copy.deepcopy(snapshot)\n\n\ndef rollout(snapshot, seat, selected, trial_seed):\n    state, actors = copy_snapshot(snapshot)\n    # Preserve each actor\'s public/private memory at the sampled decision while\n    # integrating future stochastic choices. Each action gets the same streams.\n    for player, actor in enumerate(actors):\n        if hasattr(actor, \'rng\'):\n            actor.rng = random.Random(trial_seed * 10 + player)\n    decision = engine.next_decision(state)\n    if decision.seat != seat:\n        raise RuntimeError(\'sampled decision is for a different player\')\n    if not engine.step(state, decision, engine.decode_action(state, decision, selected)).ok:\n        raise RuntimeError(\'counterfactual action is not legal at sampled state\')\n    while state.outcome is None and seat not in state.finished_set:\n        advance(state, actors)\n    return reward(state, seat)\n\n\ndef checkpoint_spec(document):\n    if document.get(\'format\') == \'triad-bluff-residual-mlp-v1\':\n        return {\'kind\': \'mlp\', \'model\': document}\n    kind = document.get(\'training\', {}).get(\'policy_kind\',\n        \'v2\' if len(document[\'weights\']) == 30 else \'v1\')\n    return {\'kind\': kind, \'weights\': document[\'weights\']}\n\n\ndef pool_specs(incumbent, previous):\n    return [checkpoint_spec(incumbent), checkpoint_spec(previous),\n            *({\'kind\': kind} for kind in (\'starter\', \'competitive\', \'random\', \'aggressive\', \'heuristic\')),\n            *({\'kind\': \'bluffer\', \'bluff_rate\': bluff, \'call_rate\': call}\n              for bluff, call in ((.1, .0), (.1, .35), (.35, .1), (.65, .35), (.95, .0), (.95, .35)))]\n\n\nSOURCE_REGIMES = (\'league\', \'neural_selfplay\', \'v2_selfplay\', \'competitive\',\n                  \'truthful\', \'bluffer\', \'frequent_call\')\n\n\ndef source_specs(regime, rng, incumbent, previous):\n    if regime == \'league\':\n        return [rng.choice(pool_specs(incumbent, previous)) for _ in range(4)]\n    if regime == \'neural_selfplay\':\n        rival = checkpoint_spec(incumbent)\n    elif regime == \'v2_selfplay\':\n        rival = checkpoint_spec(previous)\n    elif regime in (\'competitive\', \'truthful\'):\n        rival = {\'kind\': regime}\n    elif regime == \'bluffer\':\n        rival = {\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}\n    elif regime == \'frequent_call\':\n        rival = {\'kind\': \'bluffer\', \'bluff_rate\': .65, \'call_rate\': .35}\n    else:\n        raise ValueError(\'unknown source composition\')\n    return [rival] * 4\n\n\ndef collect_job(job):\n    seed, incumbent, previous, samples_per_phase, trials, max_actions, regime = job\n    rng = random.Random(seed ^ 0x424C554646)\n    own = seed % 4\n    specs = source_specs(regime, rng, incumbent, previous)\n    specs[own] = checkpoint_spec(incumbent)\n    actors = [make_policy(spec, seed * 10 + player) for player, spec in enumerate(specs)]\n    state = engine.new_game(engine.GameConfig(max_turns=1000), seed)\n    reservoirs = {\'PLACE\': [], \'CHALLENGE\': []}\n    eligible = Counter()\n    initial = incumbent.get(\'base_weights\', incumbent.get(\'weights\'))\n    feature_weights = initial + [0.] * (len(FEATURE_NAMES) - len(initial))\n    while state.outcome is None and own not in state.finished_set:\n        decision = engine.next_decision(state)\n        if decision.seat == own:\n            observation = observe_read_only(state, own)\n            policy = Features(feature_weights); policy.update(observation)\n            if len(policy.candidates(observation)) > 1:\n                phase = observation[\'phase\']\n                eligible[phase] += 1\n                index = rng.randrange(eligible[phase])\n                if index < samples_per_phase:\n                    snapshot = copy_snapshot((state, actors))\n                    if len(reservoirs[phase]) < samples_per_phase:\n                        reservoirs[phase].append(snapshot)\n                    else:\n                        reservoirs[phase][index] = snapshot\n        advance(state, actors)\n    rows = []\n    for phase in (\'PLACE\', \'CHALLENGE\'):\n        for snapshot in reservoirs[phase]:\n            original, _ = snapshot\n            observation = observe_read_only(original, own)\n            policy = Features(feature_weights); policy.update(observation)\n            candidates = policy.candidates(observation)\n            baseline = make_policy(specs[own], seed * 10 + own).act(observation)\n            preferred = next(item for item in candidates if item[0] == baseline) if any(item[0] == baseline for item in candidates) else candidates[0]\n            # Identical model inputs cannot be distinguished by the scorer;\n            # avoid spending rollout compute on exact feature aliases.\n            representatives = {tuple(preferred[1]): preferred}\n            for candidate in candidates:\n                representatives.setdefault(tuple(candidate[1]), candidate)\n            selected = list(representatives.values())\n            if len(selected) > max_actions:\n                selected = [preferred] + rng.sample([item for item in selected if item is not preferred], max_actions - 1)\n            trial_values = [[rollout(snapshot, own, action, seed * 100_000 + len(rows) * 1000 + trial)\n                             for trial in range(trials)] for action, _ in selected]\n            features = [vector for _, vector in selected]\n            targets = [sum(values) / trials for values in trial_values]\n            rows.append((features, targets, trial_values, seed, phase, len(observation[\'hand\']), regime))\n    return rows\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path, required=True)\n    parser.add_argument(\'--previous\', type=Path, required=True)\n    parser.add_argument(\'--games\', type=int, default=2000)\n    parser.add_argument(\'--samples-per-phase\', type=int, default=2)\n    parser.add_argument(\'--trials\', type=int, default=2)\n    parser.add_argument(\'--max-actions\', type=int, default=10)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    parser.add_argument(\'--seed-start\', type=int, default=50_000_000)\n    parser.add_argument(\'--source-compositions\', choices=(\'league\', \'balanced\'), default=\'league\')\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    if min(args.games, args.samples_per_phase, args.trials, args.workers) < 1 or args.max_actions < 2:\n        parser.error(\'positive counts and at least two candidate actions required\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    incumbent = json.loads(args.incumbent.read_text()); previous = json.loads(args.previous.read_text())\n    features, targets, trial_targets, offsets, game_ids = [], [], [], [0], []\n    phases = Counter(); hands = Counter(); compositions = Counter(); source_regimes = []; started = time.time()\n    jobs = [(seed, incumbent, previous, args.samples_per_phase, args.trials, args.max_actions,\n             SOURCE_REGIMES[(seed - args.seed_start) % len(SOURCE_REGIMES)] if args.source_compositions == \'balanced\' else \'league\')\n            for seed in range(args.seed_start, args.seed_start + args.games)]\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        for index, rows in enumerate(pool.map(collect_job, jobs)):\n            for vectors, values, trials, seed, phase, hand, regime in rows:\n                features.extend(vectors); targets.extend(values); trial_targets.extend(trials)\n                offsets.append(len(features)); game_ids.append(seed); phases[phase] += 1; hands[hand] += 1\n                source_regimes.append(SOURCE_REGIMES.index(regime)); compositions[regime] += 1\n            if (index + 1) % 100 == 0:\n                print(json.dumps({\'games\': index + 1, \'states\': len(game_ids), \'actions\': len(features),\n                                  \'seconds\': round(time.time() - started, 1)}), flush=True)\n    np.savez_compressed(args.output / \'counterfactual.npz\', features=np.asarray(features, dtype=np.float32),\n                        targets=np.asarray(targets, dtype=np.float32), trial_targets=np.asarray(trial_targets, dtype=np.float32),\n                        offsets=np.asarray(offsets, dtype=np.int64), game_ids=np.asarray(game_ids, dtype=np.int64),\n                        source_regimes=np.asarray(source_regimes, dtype=np.int8))\n    metadata = {\'method\': \'paired one-step policy improvement, phase-stratified whole-game reservoirs\',\n                \'input_features\': FEATURE_NAMES, \'feature_policy\': \'v5\', \'games\': args.games,\n                \'states\': len(game_ids), \'actions\': len(features), \'trials\': args.trials,\n                \'samples_per_phase\': args.samples_per_phase, \'max_actions\': args.max_actions,\n                \'seed_start\': args.seed_start, \'incumbent\': incumbent, \'opponent_pool\': pool_specs(incumbent, previous),\n                \'sdk_game_commit\': \'96da8042c285af4e4f43655f2ad99592bd49b8e6\',\n                \'incumbent_policy_kind\': checkpoint_spec(incumbent)[\'kind\'],\n                \'source_compositions\': args.source_compositions, \'source_regime_names\': SOURCE_REGIMES,\n                \'source_game_composition_histogram\': dict(Counter(job[-1] for job in jobs)),\n                \'sampled_state_composition_histogram\': dict(compositions),\n                \'fast_snapshots\': os.environ.get(\'BLUFF_FAST_SNAPSHOTS\') == \'1\',\n                \'hidden_state_use\': \'training returns only; dealt hands fixed within state, common future actor seeds across actions\',\n                \'future_draw_pile_shuffle\': False, \'own_hand_size_histogram\': dict(sorted(hands.items())),\n                \'phase_histogram\': dict(phases), \'seconds\': round(time.time() - started, 1)}\n    (args.output / \'metadata.json\').write_text(json.dumps(metadata, indent=2) + \'\\n\')\n    print(json.dumps({key: value for key, value in metadata.items() if key not in (\'incumbent\', \'input_features\', \'opponent_pool\')}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'select_bluff_residual_cpu.py': '"""CPU actual-game selection for GPU-exported neural candidates."""\nimport argparse\nimport concurrent.futures\nimport json\nimport hashlib\nimport multiprocessing\nimport os\nfrom pathlib import Path\nfrom train_bluff_league import evaluate_job, paired_interval, summarize\nfrom evaluate_bluff_checkpoint import combine\nfrom collect_bluff_counterfactual import checkpoint_spec\nfrom bluff_mlp_policy import Policy\n\n\ndef evaluate(pool, specifications, seeds, opponents, mode):\n    chunks=[seeds[start:start+16] for start in range(0,len(seeds),16)]\n    jobs=[(spec,chunk,opponents,mode) for spec in specifications for chunk in chunks]\n    raw=list(pool.map(evaluate_job,jobs));n=len(chunks)\n    return [combine(raw[index*n:(index+1)*n]) for index in range(len(specifications))]\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--candidates\',type=Path,required=True)\n    parser.add_argument(\'--linear-backup\',type=Path,required=True)\n    parser.add_argument(\'--previous-v1\',type=Path)\n    parser.add_argument(\'--archive\',type=Path,action=\'append\',default=[],help=\'additional fixed learned rivals\')\n    parser.add_argument(\'--strong-rival\',type=Path,help=\'previously verified stronger own-learned checkpoint\')\n    parser.add_argument(\'--validation-blocks\',type=int,default=128)\n    parser.add_argument(\'--test-blocks\',type=int,default=256)\n    parser.add_argument(\'--validation-seed-start\',type=int,default=65_000_000)\n    parser.add_argument(\'--test-seed-start\',type=int,default=66_000_000)\n    parser.add_argument(\'--workers\',type=int,default=4)\n    parser.add_argument(\'--robust-selection\',action=\'store_true\',help=\'select using five fixed rival compositions with a per-regime loss guard\')\n    parser.add_argument(\'--robust-calling-styles\',action=\'store_true\',help=\'also guard both fixed frequent-calling bluff styles during robust selection\')\n    parser.add_argument(\'--robust-strong-learned\',action=\'store_true\',help=\'also guard learned-only mixtures and the fixed strong rival selfplay\')\n    parser.add_argument(\'--max-selection-regression\',type=float,default=.025)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args()\n    if min(args.validation_blocks,args.test_blocks,args.workers)<1:parser.error(\'positive counts required\')\n    os.environ[\'BLUFF_FAST_OBSERVATIONS\']=\'1\';os.environ[\'OPENBLAS_NUM_THREADS\']=\'1\'\n    args.output.mkdir(parents=True,exist_ok=True)\n    fit=json.loads((args.candidates/\'fit-summary.json\').read_text())\n    incumbent_document=fit[\'baseline_checkpoint\'];incumbent=checkpoint_spec(incumbent_document)\n    linear_document=json.loads(args.linear_backup.read_text());linear=checkpoint_spec(linear_document)\n    documents=[json.loads((args.candidates/f\'candidate-{index}.json\').read_text()) for index in range(fit[\'candidate_count\'])]\n    for document in documents:Policy(document)\n    reference=[{\'kind\':kind} for kind in (\'starter\',\'competitive\',\'heuristic\',\'random\',\'truthful\',\'aggressive\')]+[incumbent,linear]\n    if args.previous_v1:reference.append(checkpoint_spec(json.loads(args.previous_v1.read_text())))\n    reference += [checkpoint_spec(json.loads(path.read_text())) for path in args.archive]\n    strong_rival=checkpoint_spec(json.loads(args.strong_rival.read_text())) if args.strong_rival else None\n    if strong_rival:reference.append(strong_rival)\n    reference += [{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':call} for bluff,call in ((.1,.0),(.1,.35),(.35,.1),(.65,.35),(.95,.0),(.95,.35))]\n    specs=[incumbent]+[checkpoint_spec(document) for document in documents]\n    seeds=list(range(args.validation_seed_start,args.validation_seed_start+args.validation_blocks))\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        results=evaluate(pool,specs,seeds,reference,\'league\')\n        selection_rows={\'league\':results}\n        chosen=max(range(len(results)),key=lambda index:results[index][\'points\'])\n        selection={\'selected\':chosen,\'seed_start\':args.validation_seed_start,\'scores\':[summarize(row) for row in results],\'paired_gains\':[paired_interval(row,results[0]) for row in results[1:]],\'opponent_pool\':reference}\n        if args.robust_selection:\n            # These seeds select a fixed checkpoint only. Fresh final seeds are\n            # never consulted during candidate or objective selection.\n            regime_results=selection_rows\n            for label,opponents,mode in [(\'neural_selfplay\',[incumbent],\'league\'),\n                                         (\'competitive\',reference,\'competitive\'),\n                                         (\'truthful\',reference,\'truthful\'),\n                                         (\'bluffer\',reference,\'bluffer\')]:\n                regime_results[label]=evaluate(pool,specs,seeds,opponents,mode)\n                print(json.dumps({\'selection_regime\':label,\'scores\':[summarize(row) for row in regime_results[label]]}),flush=True)\n            if args.robust_calling_styles:\n                for label,bluff in [(\'weakspot_bluff01_call035\',.1),(\'weakspot_bluff065_call035\',.65)]:\n                    regime_results[label]=evaluate(pool,specs,seeds,[{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':.35}],\'league\')\n                    print(json.dumps({\'selection_regime\':label,\'scores\':[summarize(row) for row in regime_results[label]]}),flush=True)\n            if args.robust_strong_learned:\n                if strong_rival is None:parser.error(\'strong learned selection requires --strong-rival\')\n                learned=[spec for spec in reference if spec[\'kind\'] in (\'mlp\',\'v1\',\'v2\',\'v3\',\'v4\',\'v5\')]\n                for label,opponents in [(\'learned_mixture\',learned),(\'strong_rival_selfplay\',[strong_rival])]:\n                    regime_results[label]=evaluate(pool,specs,seeds,opponents,\'league\')\n                    print(json.dumps({\'selection_regime\':label,\'scores\':[summarize(row) for row in regime_results[label]]}),flush=True)\n            gains={label:[summarize(row)[\'points\']-summarize(rows[0])[\'points\'] for row in rows]\n                   for label,rows in regime_results.items()}\n            utilities=[sum(values[index] for values in gains.values())/len(gains) for index in range(len(specs))]\n            feasible=[index for index in range(len(specs))\n                      if min(values[index] for values in gains.values())>=-args.max_selection_regression]\n            chosen=max(feasible,key=lambda index:utilities[index])\n            selection.update(selected=chosen,selection_rule=\'equal mean gain across predeclared regimes, subject to per-regime mean regression guard; baseline wins ties\',\n                             selection_regimes=list(regime_results),\n                             max_selection_regression=args.max_selection_regression,\n                             regime_scores={label:[summarize(row) for row in rows] for label,rows in regime_results.items()},\n                             regime_paired_gains={label:[paired_interval(row,rows[0]) for row in rows[1:]] for label,rows in regime_results.items()},\n                             feasible_candidates=feasible,mean_gain_utilities=utilities)\n        (args.output/\'selection.json\').write_text(json.dumps(selection,indent=2)+\'\\n\')\n        identity={\'sdk_game_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\',\n                  \'game_config\':{\'players\':4,\'decks\':1,\'max_turns\':1000,\'position_points\':[5,3,2,1]},\n                  \'opponent_pool_canonical_sha256\':hashlib.sha256(json.dumps(reference,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest(),\n                  \'checkpoint_canonical_sha256\':[hashlib.sha256(json.dumps(document,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest() for document in [incumbent_document]+documents],\n                  \'sampling_unit\':\'one seed, all four seats averaged; ordered seed_ids match each block array\'}\n        (args.output/\'selection-blocks.json\').write_text(json.dumps({**identity,\'seed_ids\':seeds,\'rows\':selection_rows},indent=2)+\'\\n\')\n        print(json.dumps({\'actual_game_selection\':{key:value for key,value in selection.items() if key!=\'opponent_pool\'}}),flush=True)\n        selected_document=documents[chosen-1] if chosen else incumbent_document\n        (args.output/\'bluff-policy-mlp.json\').write_text(json.dumps(selected_document,indent=2)+\'\\n\')\n        selected=specs[chosen];final_seeds=list(range(args.test_seed_start,args.test_seed_start+args.test_blocks));report={};final_rows={}\n        regimes=[(\'league\',reference,\'league\'),(\'competitive\',reference,\'competitive\'),(\'starter\',reference,\'starter\'),(\'truthful\',reference,\'truthful\'),(\'bluffer\',reference,\'bluffer\'),(\'aggressive\',reference,\'aggressive\'),(\'incumbent_selfplay\',[incumbent],\'league\')]\n        regimes += [(label,[{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':.35}],\'league\') for label,bluff in ((\'weakspot_bluff01_call035\',.1),(\'weakspot_bluff065_call035\',.65))]\n        if args.robust_strong_learned:\n            regimes += [(\'learned_mixture\',[spec for spec in reference if spec[\'kind\'] in (\'mlp\',\'v1\',\'v2\',\'v3\',\'v4\',\'v5\')],\'league\'),(\'strong_rival_selfplay\',[strong_rival],\'league\')]\n        for label,opponents,mode in regimes:\n            rows=evaluate(pool,[selected,incumbent,linear],final_seeds,opponents,mode)\n            final_rows[label]={\'candidate\':rows[0],\'incumbent_neural\':rows[1],\'retained_v2\':rows[2]}\n            report[label]={\'candidate\':summarize(rows[0]),\'incumbent_neural\':summarize(rows[1]),\'retained_v2\':summarize(rows[2]),\'paired_gain_vs_neural\':paired_interval(rows[0],rows[1]),\'paired_gain_vs_v2\':paired_interval(rows[0],rows[2])}\n            print(json.dumps({\'final_regime\':label,**report[label]}),flush=True)\n        (args.output/\'evaluation.json\').write_text(json.dumps({\'seed_start\':args.test_seed_start,\'candidate_training\':selected_document[\'training\'],\'baseline_training\':incumbent_document[\'training\'],\'evaluation\':report},indent=2)+\'\\n\')\n        final_identity={\'candidate\':identity[\'checkpoint_canonical_sha256\'][chosen],\n                        \'incumbent_neural\':identity[\'checkpoint_canonical_sha256\'][0],\n                        \'retained_v2\':hashlib.sha256(json.dumps(linear_document,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()}\n        (args.output/\'evaluation-blocks.json\').write_text(json.dumps({**identity,\'seed_ids\':final_seeds,\'selected_index\':chosen,\'final_checkpoint_canonical_sha256\':final_identity,\'rows\':final_rows},indent=2)+\'\\n\')\n\n\nif __name__==\'__main__\':main()\n', 'evaluate_bluff_checkpoint.py': '"""Benchmark trained Bluff against actual SDK starter and adaptive mixtures."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nfrom pathlib import Path\nfrom train_bluff_league import evaluate_job, paired_interval, summarize\nfrom bluff_policy import INITIAL_WEIGHTS\nfrom bluff_policy_v2 import INITIAL_WEIGHTS as V2_INITIAL\n\n\ndef combine(parts):\n points=[v for p in parts for v in p[\'block_points\']]\n wins=[v for p in parts for v in p[\'block_wins\']]\n return {\'points\':sum(points)/len(points),\'win_rate\':sum(wins)/len(wins),\'games\':4*len(points),\'block_points\':points,\'block_wins\':wins}\n\n\ndef main():\n parser=argparse.ArgumentParser()\n parser.add_argument(\'--model\',type=Path,required=True)\n parser.add_argument(\'--blocks\',type=int,default=256)\n parser.add_argument(\'--workers\',type=int,default=4)\n parser.add_argument(\'--output\',type=Path,default=Path(__file__).parent/\'bluff-v1-verification.json\')\n args=parser.parse_args()\n document=json.loads(args.model.read_text())\n candidate={\'kind\':\'v1\',\'weights\':document[\'weights\']};starter={\'kind\':\'starter\'}\n reference=[{\'kind\':k} for k in (\'starter\',\'heuristic\',\'truthful\',\'aggressive\',\'random\')]+[\n  {\'kind\':\'v1\',\'weights\':INITIAL_WEIGHTS},{\'kind\':\'v2\',\'weights\':V2_INITIAL},candidate]+[\n  {\'kind\':\'bluffer\',\'bluff_rate\':b,\'call_rate\':c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n modes=(\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\')\n seeds=list(range(13_000_000,13_000_000+args.blocks));chunks=[seeds[i:i+32] for i in range(0,len(seeds),32)]\n jobs=[(spec,chunk,reference,mode) for mode in modes for spec in (candidate,starter) for chunk in chunks]\n with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n  results=list(pool.map(evaluate_job,jobs))\n report={}\n for index,mode in enumerate(modes):\n  size=len(chunks);a=combine(results[(2*index)*size:(2*index+1)*size]);b=combine(results[(2*index+1)*size:(2*index+2)*size])\n  report[mode]={\'trained\':summarize(a),\'sdk_starter\':summarize(b),\'paired_gain\':paired_interval(a,b)}\n payload={\'model_training\':document[\'training\'],\'seed_start\':seeds[0],\'sampling_unit\':\'four-seat seed block\',\'evaluation\':report}\n args.output.write_text(json.dumps(payload,indent=2)+\'\\n\');print(json.dumps(payload),flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'evaluate_bluff_threeway.py': '"""Fresh fixed-pool paired gate against retained checkpoints."""\nimport argparse\nimport concurrent.futures\nimport json\nimport hashlib\nimport multiprocessing\nimport os\nfrom pathlib import Path\nfrom collect_bluff_counterfactual import checkpoint_spec\nfrom select_bluff_residual_cpu import evaluate\nfrom train_bluff_league import paired_interval,summarize\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--candidate\',type=Path,required=True)\n    parser.add_argument(\'--incumbent\',type=Path,default=Path(\'bluffSDK/models/policy.json\'))\n    parser.add_argument(\'--v2\',type=Path,default=Path(\'bluffSDK/models/policy-v2.json\'))\n    parser.add_argument(\'--blocks\',type=int,default=512)\n    parser.add_argument(\'--seed-start\',type=int,required=True)\n    parser.add_argument(\'--workers\',type=int,default=3)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--opponent-pool\',type=Path,help=\'reuse a frozen selection pool JSON\')\n    parser.add_argument(\'--strong-rival\',type=Path,help=\'also compare the fixed conservative candidate and its selfplay regime\')\n    parser.add_argument(\'--robust-strong-learned\',action=\'store_true\',help=\'include the frozen pool learned-only mixture\')\n    parser.add_argument(\'--regime\',action=\'append\',default=[],help=\'run only named predefined regimes; repeat to select more than one\')\n    args=parser.parse_args();os.environ[\'BLUFF_FAST_OBSERVATIONS\']=\'1\';os.environ[\'OPENBLAS_NUM_THREADS\']=\'1\'\n    paths=[args.candidate,args.incumbent,args.v2]\n    labels=[\'candidate\',\'incumbent_neural\',\'retained_v2\']\n    if args.strong_rival:\n        paths.append(args.strong_rival);labels.append(\'conservative_candidate\')\n    docs=[json.loads(path.read_text()) for path in paths];specs=[checkpoint_spec(doc) for doc in docs]\n    reference=[{\'kind\':kind} for kind in (\'starter\',\'competitive\',\'heuristic\',\'random\',\'truthful\',\'aggressive\')]+specs[1:]\n    reference += [checkpoint_spec(json.loads(Path(\'bluffSDK/models/policy-v1.json\').read_text()))]\n    reference += [{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':call} for bluff,call in ((.15,.03),(.35,.1),(.5,.35))]\n    if args.opponent_pool:\n        frozen=json.loads(args.opponent_pool.read_text())\n        reference=frozen[\'opponent_pool\'] if isinstance(frozen,dict) else frozen\n        if not isinstance(reference,list) or not reference:\n            raise ValueError(\'frozen opponent pool must be a nonempty list\')\n    regimes=[(mode,reference,mode) for mode in (\'league\',\'competitive\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\')]\n    regimes += [(\'neural_selfplay\',[specs[1]],\'league\'),(\'v2_selfplay\',[specs[2]],\'league\')]\n    regimes += [(label,[{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':.35}],\'league\') for label,bluff in ((\'weakspot_bluff01_call035\',.1),(\'weakspot_bluff065_call035\',.65))]\n    if args.robust_strong_learned:\n        learned=[row for row in reference if row[\'kind\'] in (\'mlp\',\'v1\',\'v2\',\'v3\',\'v4\',\'v5\')]\n        if not learned:parser.error(\'strong learned regime needs learned checkpoints in the frozen pool\')\n        regimes.append((\'learned_mixture\',learned,\'league\'))\n    if args.strong_rival:\n        regimes.append((\'strong_rival_selfplay\',[specs[3]],\'league\'))\n    if args.regime:\n        unknown=set(args.regime)-{row[0] for row in regimes}\n        if unknown:parser.error(\'unknown regimes: \'+\',\'.join(sorted(unknown)))\n        regimes=[row for row in regimes if row[0] in args.regime]\n    seeds=list(range(args.seed_start,args.seed_start+args.blocks));report={};block_rows={}\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        for label,opponents,mode in regimes:\n            rows=evaluate(pool,specs,seeds,opponents,mode)\n            block_rows[label]=dict(zip(labels,rows))\n            report[label]={name:summarize(row) for name,row in zip(labels,rows)}\n            report[label].update({\'paired_gain_vs_neural\':paired_interval(rows[0],rows[1]),\'paired_gain_vs_v2\':paired_interval(rows[0],rows[2])})\n            if args.strong_rival:\n                report[label][\'paired_gain_vs_conservative\']=paired_interval(rows[0],rows[3])\n            print(json.dumps({\'mode\':label,**report[label]}),flush=True)\n    args.output.write_text(json.dumps({\'seed_start\':args.seed_start,\'opponent_pool\':reference,\'candidate_training\':docs[0][\'training\'],\'evaluation\':report},indent=2)+\'\\n\')\n    sidecar={\'sdk_game_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\',\n             \'game_config\':{\'players\':4,\'decks\':1,\'max_turns\':1000,\'position_points\':[5,3,2,1]},\n             \'seed_ids\':seeds,\'sampling_unit\':\'one seed, all four seats averaged; ordered seed_ids match each block array\',\n             \'checkpoint_order\':labels,\n             \'checkpoint_file_sha256\':[hashlib.sha256(path.read_bytes()).hexdigest() for path in paths],\n             \'checkpoint_canonical_sha256\':[hashlib.sha256(json.dumps(document,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest() for document in docs],\n             \'opponent_pool_canonical_sha256\':hashlib.sha256(json.dumps(reference,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest(),\n             \'rows\':block_rows}\n    args.output.with_name(args.output.stem+\'-blocks.json\').write_text(json.dumps(sidecar,indent=2)+\'\\n\')\n\n\nif __name__==\'__main__\':main()\n', 'bluff_job_deadline.py': '"""Stop future private job subprocess groups before the event packaging window."""\nfrom datetime import datetime,timezone\nimport os\nimport signal\nimport subprocess\nimport time\n\nOFFICIAL_FREEZE_UTC=\'2026-10-15T07:30:00+00:00\'\nJOB_STOP_UTC=\'2026-10-15T06:00:00+00:00\' # 90 minutes for download, review, packaging and push.\nJOB_STOP_TIMESTAMP=datetime.fromisoformat(JOB_STOP_UTC).timestamp()\n\n\ndef run_before_deadline(command,maximum_seconds,*,stop_timestamp=JOB_STOP_TIMESTAMP):\n    """Use a new process group so timeout also stops process-pool descendants."""\n    remaining=stop_timestamp-time.time()\n    # Reserve kill grace inside the absolute cutoff. A late queued notebook\n    # never obtains a new six-hour allowance extending beyond the event freeze.\n    if remaining<=5:\n        raise TimeoutError(\'Bluff preparation cutoff reached; preserve the verified incumbent.\')\n    limit=min(float(maximum_seconds),remaining-5)\n    if limit<=0:raise ValueError(\'positive per-process time budget required\')\n    process=subprocess.Popen(command,start_new_session=True)\n    try:\n        code=process.wait(timeout=limit)\n        if code:raise subprocess.CalledProcessError(code,command)\n    except BaseException:\n        try:os.killpg(process.pid,signal.SIGTERM)\n        except ProcessLookupError:pass\n        try:process.wait(timeout=3)\n        except subprocess.TimeoutExpired:pass\n        # The leader can exit before a descendant; kill the entire original\n        # group even when the immediate child\'s return code is already known.\n        try:os.killpg(process.pid,signal.SIGKILL)\n        except ProcessLookupError:pass\n        process.wait()\n        raise\n    return subprocess.CompletedProcess(command,code)\n'}
for name,text in sources.items(): (work/name).write_text(text)
sys.path.insert(0,str(work))
from bluff_job_deadline import run_before_deadline
sdk=Path("/kaggle/working/bluffSDK")
run_before_deadline(["git","clone","--quiet","https://github.com/Sir-Nike/bluffSDK",str(sdk)],120)
run_before_deadline(["git","-C",str(sdk),"checkout","--quiet","96da8042c285af4e4f43655f2ad99592bd49b8e6"],120)
os.environ["BLUFF_SDK"]=str(sdk)
output=Path("/kaggle/working/output");output.mkdir(exist_ok=True)
(work/'incumbent.json').write_bytes(zlib.decompress(base64.b64decode('...')))
(work/'v2.json').write_bytes(zlib.decompress(base64.b64decode('eJx9VcuO4zgMvPdXBDlvDImiXvsri4Wh2HIitGwZfmTQPZh/H9qK3UlndgPk4CJVLrJI+efb4XBs0tC66fj34TgNwdWnc5yb5hRD591wusHxrzXJu2ke/Ehp/9AzIWc3VddyDJ9+zTgsx+fp2sxxe16Jnh7KPkT/jFQuxnJw0ze4CV0Yrxs2+NYR0F3Ky5DmfnzFoxsufpxyfAtX0YXW10TfvZdXH+v9HcE9cNy8i5SVX/wo5eXcSlhWae6mDVoqeupC6vvU+W66l0Di9siPrhzbpd6r6+qX/NfQ6No++vG5M677yE3rh3R25xDD9PGcEdo+jWM4kzDydk32w3eTSrKzG0vqYLr5+pngvVukrpU1g6umkLqX488G5XNd2t+9yrrrvIvZOxZd5YfyEkk+Wf/f7c8y0jyNofbZxO9y/kdp9mpDy9SUaQiX0LnnHu8u5REq17H+Rp8FrwLOnlrqy5X8SEn/rtvxw4fLdfpaDlagltIKbgGBc25FJpSFAaa0Qs20thpUhk+iQJQWGCojJbNbOiskAykYlxyMRW71PZ+zgpKtAU1hoxHtV0AJpYH+nEluIOO8sBzACCsoCoiAGVeFsqC50fQO0oMyw6IQwhphKE8bg4h612mVtlyARauM4XdcF9wwS0IsUwZ2MaYANFSrFEBalOB3HAuQFDEknwEd2dhlobkWliFo4hJCPVRrtZYcqXXUBq52Ik4NIApkXJCCXb5CQU0UVKjlbKsKC20kE4LkIDI0epdpmUWmpNJacL3BlooCZbi1in5WbdzkK5NARIppwmETWUghyC1lUCPhenPcSs2EARBs8cRk2FKpSMVyYjLImPxiNyBoNoTm9M698ctAGSZomAwZI42We3MKUMKiQq6A6cWYbXYUV+QhI9tpHqg/GYfCqGUsqfmK6t4m5EQecqmBZk0KybSFfbinId+xNN0/81q0frqmevliuKG6hps/Re8usz9UA+36iS6zIfUfhz7FUH0cUj+FNny6x+UcvV/OAwPF2ebn8eI7P6x5yyoJtl0YqZ9jPk4o7DcRyaJbI1XvS7Z5gq+0659rPrCNxt9cnFeahzBne5zCoc7xnVXhLjj6aqJr6iGrT6Fblx4LZmjb9d3FY+iquT0vd/qfsmk8gbOH9LF+p29K24b1I2xV7QztQAVGugY9NiiUlA242lq6JM412rPxarl9fr39evsNaNwPdg==')))
(work/'v1.json').write_bytes(zlib.decompress(base64.b64decode('eJxlVElu4zAQvOcVgs9jgTub/kowEGiJsohQCyjKQRLk79NanYxPFKu7qlfq6yXLTnUfW5tOl+yUorfV+Rqmuj4H3zkbz3d6+rM4OZum6EZ0e8U7IlebyqYY/adbPLKZPqWmnsJ+X4R+XYrBB/cbKW0IRbTpP7j2nR+bHYuutQh0t+IW+2kYn/Fg482NabXv5jJY37oK5bu3onGhOmJ4+0Pj7mxArzXwz1SeeItgUfZTl3ZoruhXF/ph6DvXpa0ETO6wvHfF2M71NrarnvyfTaNth4A9x9vfZQrvzt+a9BiCyLmkoAA4V6CEYStR5oozpilTeAoQVK74meWKKQKaCwP4RZVeDSTXRHHDmSZE4KnMRqAkRzpIoFJKBmwXgpwToiXR2oAhwMWGo5AApSnlnCPK9EagOcPAEjhlDIQAUHuAXEotMSxCmiFrK+Es0ABUgyJKG0o4X3HIGfpjkoQDmuWev+BYlSGUSWMEyN0d9bmUijLBiOGECgKHYc5RMoNpGkzgIUWFVooINCuOTRLykSo2jmNkgZLYdHkULTQnhlEKlBijjuJIjiVQQyk2ynCJPT8mmeK6uDjKr3XYrUtNX83PsIz9OJ5xJWI/fGRDH3z5kfVD8q3/tMn33bEfzs0ERnCUWNuG3lzn4uI3L8o+stPQD1NY6Zdsb/PpZls3FoOL+BC7ylfz9l8ytWvdbZixjZQD7hPQQ3Ks3vAxtK1f/h5GVRaIYCUDaWvhRC24krJmtjJGGnathLmCU/M6f798v/wDMqoe+g==')))
(work/'strong-rival.json').write_bytes(zlib.decompress(base64.b64decode('...')))
(work/"frozen-pool.json").write_bytes(zlib.decompress(base64.b64decode('...')))
candidates=list(Path("/kaggle/input").rglob("bluff-policy-mlp.json"))
assert len(candidates)==1,"ambiguous selected checkpoint"
candidate=candidates[0]
assert hashlib.sha256(candidate.read_bytes()).hexdigest()=='8253987d638222d8f1314baa5e7045f1b5ab58c5b224506be011078e21d330ba'
(sdk/"models").mkdir(exist_ok=True)
(sdk/"models/policy-v1.json").write_bytes((work/"v1.json").read_bytes())
run_before_deadline(["python",str(work/"evaluate_bluff_threeway.py"),"--candidate",str(candidate),"--incumbent",str(work/"incumbent.json"),"--v2",str(work/"v2.json"),"--strong-rival",str(work/"strong-rival.json"),"--robust-strong-learned","--opponent-pool",str(work/"frozen-pool.json"),"--blocks","512","--seed-start","90000000","--workers","4","--output",str(output/"independent.json")],21600)
